# Databricks Certified Data Engineer Associate Exam Cheatsheet

## Page 1: Core Concepts & Architecture

### Databricks Lakehouse Platform
- **Lakehouse**: Combines data lake storage with data warehouse capabilities
- **Delta Lake**: Open-source storage layer providing ACID transactions
- **Unity Catalog**: Unified governance solution for data and AI assets

### Cluster Types
- **All-Purpose Clusters**: Interactive analysis, shared among users
- **Job Clusters**: Automated jobs, terminated after completion (cost-effective)
- **Cluster Modes**: Standard (multi-node), Single Node, High Concurrency

### Delta Lake Features
```python
# Create Delta table
df.write.format("delta").save("/path/to/table")

# Time travel
df = spark.read.format("delta").option("versionAsOf", 0).load("/path")
df = spark.read.format("delta").option("timestampAsOf", "2024-01-01").load("/path")

# OPTIMIZE and Z-ORDER
OPTIMIZE table_name ZORDER BY (column1, column2)

# VACUUM (remove old files)
VACUUM table_name RETAIN 168 HOURS
```

### Databricks File System (DBFS)
- **Paths**: `/dbfs/`, `dbfs:/`, `/mnt/`
- **dbutils.fs**: `ls()`, `rm()`, `mv()`, `cp()`, `mkdirs()`

### Data Formats
- **Parquet**: Columnar, compressed, default for Delta
- **JSON**: Semi-structured data
- **CSV**: Text-based, row-oriented
- **Avro**: Row-based, schema evolution

---

## Page 2: ETL & Data Processing

### DataFrames & Spark SQL
```python
# Read data
df = spark.read.format("json").load("/path")
df = spark.read.table("catalog.schema.table")

# Transformations
df.select("col1", "col2").filter(col("col1") > 10)
df.groupBy("category").agg(sum("amount"), avg("price"))
df.join(df2, "id", "inner")

# Write data
df.write.mode("overwrite").saveAsTable("table_name")
df.write.format("delta").partitionBy("date").save("/path")
```

### Medallion Architecture
- **Bronze**: Raw ingested data (append-only)
- **Silver**: Cleaned, conformed data (deduplicated, validated)
- **Gold**: Business-level aggregates (analytics-ready)

### Change Data Capture (CDC)
```sql
-- MERGE for upserts
MERGE INTO target t
USING source s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

### Streaming
```python
# Structured Streaming
stream = spark.readStream.format("delta").load("/path")
query = stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/checkpoint") \
    .start("/output")

# Trigger options
.trigger(once=True)  # Single micro-batch
.trigger(processingTime='10 seconds')  # Continuous
```

### Delta Live Tables (DLT)
```python
import dlt

@dlt.table
def bronze_table():
    return spark.read.format("json").load("/source")

@dlt.table
def silver_table():
    return dlt.read("bronze_table").filter("col IS NOT NULL")

# Streaming
@dlt.table
def streaming_table():
    return dlt.read_stream("source_table")
```

---

## Page 3: Workflows, Optimization & Governance

### Databricks Workflows
- **Jobs**: Scheduled or triggered execution
- **Task Types**: Notebook, JAR, Python, SQL, dbt, Pipeline
- **Dependencies**: Task dependencies, conditional execution
- **Retries**: Configurable retry policies

### Performance Optimization
```sql
-- Partitioning
CREATE TABLE table_name 
PARTITIONED BY (year, month)
LOCATION '/path'

-- Table properties
CREATE TABLE table_name (...)
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
)

-- Caching
CACHE TABLE table_name
SELECT * FROM table_name  -- Materializes cache

-- Analyze table statistics
ANALYZE TABLE table_name COMPUTE STATISTICS
```

### Unity Catalog
```sql
-- Three-level namespace
SELECT * FROM catalog.schema.table

-- Grant permissions
GRANT SELECT ON TABLE catalog.schema.table TO user@domain.com
GRANT CREATE TABLE ON SCHEMA catalog.schema TO group_name

-- Create external location
CREATE EXTERNAL LOCATION location_name
URL 's3://bucket/path'
WITH (STORAGE CREDENTIAL credential_name)
```

### Data Quality
```python
# Expectations in DLT
@dlt.expect("valid_id", "id IS NOT NULL")
@dlt.expect_or_drop("valid_amount", "amount > 0")
@dlt.expect_or_fail("valid_date", "date >= '2020-01-01'")
```

### Best Practices
- Use **Auto Loader** for incremental file ingestion
- Enable **Auto Optimize** for small file compaction
- Use **Z-ORDER** for frequently filtered columns
- Set appropriate **retention periods** (VACUUM)
- Implement **data quality checks** early in pipeline
- Use **job clusters** for production workloads
- Monitor with **Databricks SQL queries** and dashboards

### Key SQL Commands
```sql
-- Describe table
DESCRIBE EXTENDED table_name
DESCRIBE HISTORY table_name

-- Restore version
RESTORE TABLE table_name TO VERSION AS OF 5

-- Clone tables
CREATE TABLE clone_table SHALLOW CLONE source_table
CREATE TABLE clone_table DEEP CLONE source_table

-- Drop table
DROP TABLE IF EXISTS table_name
```

---

**Exam Tips**: Focus on Delta Lake operations, streaming concepts, Unity Catalog permissions, and medallion architecture patterns.

# Databricks Certified Data Engineer Associate — Cheatsheet (≤ 3 pages)

## Page 1 — Lakehouse, Delta Lake, Tables, SQL

### Lakehouse building blocks
- **Spark**: execution engine (DataFrames + SQL).
- **Delta Lake**: storage layer on cloud object storage; adds **ACID**, **schema enforcement/evolution**, **time travel**, **MERGE**, **OPTIMIZE**, **VACUUM**.
- **Databricks SQL**: warehouses for BI/SQL workloads.
- **Unity Catalog (UC)**: governance + 3-level namespace: `catalog.schema.table`.

### Delta table essentials (SQL)
````sql
-- Create managed Delta table
CREATE TABLE main.analytics.sales (
  id BIGINT,
  dt DATE,
  amount DECIMAL(18,2)
) USING DELTA;

-- Insert / CTAS
INSERT INTO main.analytics.sales VALUES (1, '2026-01-01', 10.00);

CREATE TABLE main.analytics.sales_copy
AS SELECT * FROM main.analytics.sales;

-- History & time travel
DESCRIBE HISTORY main.analytics.sales;
SELECT * FROM main.analytics.sales VERSION AS OF 3;
SELECT * FROM main.analytics.sales TIMESTAMP AS OF '2026-01-01T00:00:00Z';

-- Restore
RESTORE TABLE main.analytics.sales TO VERSION AS OF 3;
````

### Key Delta operations
- **MERGE INTO**: upserts / CDC pattern.
- **OPTIMIZE**: compacts small files.
- **ZORDER**: co-locates data for selective queries.
- **VACUUM**: removes old files (respect retention + time travel needs).

````sql
MERGE INTO main.silver.customer t
USING main.bronze.customer_updates s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

OPTIMIZE main.silver.customer ZORDER BY (customer_id);
VACUUM main.silver.customer RETAIN 168 HOURS;
````

### Managed vs external
- **Managed table**: Databricks controls storage location.
- **External table**: data stored in your specified path / external location (common with UC + external locations).

### Common exam gotchas
- Time travel depends on **log + file retention**; VACUUM can break old versions if retention is too low.
- Partitioning helps *pruning* but can hurt if high-cardinality.
- Delta is still files under the hood → small files matter.

---

## Page 2 — Ingestion, ETL, Streaming, DLT

### Medallion architecture
- **Bronze**: raw ingest (append-only, minimal transforms).
- **Silver**: clean + dedupe + conform + enforce types.
- **Gold**: business aggregates / serving layer.

### Reading & writing (PySpark)
````python
from pyspark.sql.functions import col

df = spark.read.format("json").load("/mnt/raw/events")
clean = df.filter(col("id").isNotNull())

(clean.write
  .format("delta")
  .mode("append")
  .saveAsTable("main.bronze.events"))
````

### Auto Loader (file ingestion)
- For incremental ingest from cloud storage with schema inference/evolution.
- Uses **checkpoints** to track progress.

````python
stream_df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", "/mnt/_schemas/events")
  .load("/mnt/raw/events"))

(stream_df.writeStream
  .format("delta")
  .option("checkpointLocation", "/mnt/_checkpoints/events")
  .outputMode("append")
  .toTable("main.bronze.events_stream"))
````

### Structured Streaming basics
- **Output modes**: `append`, `update`, `complete` (depends on aggregation/watermarking).
- **Checkpointing** is required for exactly-once-ish behavior with sinks like Delta.
- **Triggers**: processing time, `availableNow`/once patterns (Databricks).

### Delta Live Tables (DLT)
- Declarative pipelines, manages dependencies + quality expectations.
- Concepts: **LIVE tables**, **streaming tables**, **expectations** (drop/fail).

````python
import dlt

@dlt.table(name="bronze_events")
def bronze_events():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("/mnt/raw/events"))

@dlt.table(name="silver_events")
@dlt.expect_or_drop("valid_id", "id IS NOT NULL")
def silver_events():
    return dlt.read_stream("bronze_events")
````

### CDC patterns
- Batch CDC: `MERGE INTO` from changes table.
- Streaming CDC: ingest changes as stream → merge into silver.

---

## Page 3 — Workflows, Performance, Governance, Reliability

### Databricks Workflows (Jobs)
- **Job clusters**: best for production (ephemeral, reproducible).
- **All-purpose clusters**: interactive (not ideal for scheduled prod).
- **Task graph**: dependencies, retries, timeouts, parameters.

### Performance checklist
- **Small files**: fix via `OPTIMIZE` and/or Auto Optimize settings.
- **Partitioning**: choose low/moderate cardinality columns frequently filtered (e.g., date).
- **ZORDER**: for high-selectivity columns used in filters/joins.
- **Stats**: help optimizer (SQL `ANALYZE TABLE … COMPUTE STATISTICS`).
- **Caching**: useful for hot data, but watch cluster memory.

````sql
ALTER TABLE main.silver.customer SET TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

ANALYZE TABLE main.silver.customer COMPUTE STATISTICS;
CACHE TABLE main.gold.customer_kpis;
````

### Unity Catalog essentials
- **3-level naming**: `catalog.schema.object`.
- **Security model**: grants on catalogs/schemas/tables/views/functions.
- **External locations + storage credentials**: controlled access to cloud storage paths.

````sql
GRANT USAGE ON CATALOG main TO `data_engineers`;
GRANT USAGE, CREATE ON SCHEMA main.silver TO `data_engineers`;
GRANT SELECT ON TABLE main.silver.customer TO `bi_users`;
````

### Reliability & operations
- **Idempotency**: design jobs so reruns don’t duplicate (MERGE, overwrite-by-partition, dedupe keys).
- **Watermarking** (streaming): handle late data; prevents unbounded state.
- **Monitoring**: job run logs, streaming query progress, table history (`DESCRIBE HISTORY`).

### Typical exam focus areas
- Delta Lake commands/behavior (MERGE, OPTIMIZE, VACUUM, time travel).
- Streaming with Auto Loader + checkpoints.
- DLT concepts + expectations.
- Unity Catalog permissions & object hierarchy.
- Job vs all-purpose clusters; orchestration fundamentals.

If you want, I can tailor this to the exact exam blueprint you’re using (Databricks has slightly different emphases across versions) and turn it into a printable 2–3 page PDF layout.

Here is a 3-page cheatsheet for the Databricks Certified Data Engineer Associate exam.

### Page 1: Core Concepts & Architecture

**Databricks Lakehouse Platform**
- **Unified Platform**: Combines Data Lakes (cheap, flexible storage) and Data Warehouses (ACID, performance).
- **Delta Lake**: The default storage format. An open-source layer over cloud storage (S3/ADLS/GCS) that provides:
    - **ACID Transactions**: Guarantees data integrity.
    - **Time Travel**: Query previous versions of data.
    - **Schema Enforcement & Evolution**: Prevents bad data writes; allows schema changes.
    - **Unified Batch & Streaming**: Use the same table for both.
- **Unity Catalog (UC)**: Centralized governance for all data and AI assets. Provides a 3-level namespace (`catalog.schema.table`), access control, and data lineage.

**Medallion Architecture**
A multi-layered approach to structure data in the lakehouse.
- **Bronze (Raw)**: Ingested raw data, append-only. Source data as-is.
- **Silver (Cleansed/Validated)**: Filtered, cleaned, and enriched data. Ready for ad-hoc analytics.
- **Gold (Aggregated)**: Business-level aggregates and features. Ready for BI and reporting.

**Compute: Clusters**
- **All-Purpose Cluster**: For interactive analysis and development. Can be shared by multiple users. Stays active until manually terminated.
- **Job Cluster**: For automated, scheduled jobs. A new cluster is created for the job run and terminates when the job is complete. **Best practice for production workloads** as it's cheaper and provides isolated environments.

**Delta Lake SQL Essentials**
````sql
-- Create a managed Delta table
CREATE TABLE main.silver.users (
  user_id LONG,
  name STRING,
  signup_date DATE
) USING DELTA
PARTITIONED BY (signup_date);

-- Query a historical version (Time Travel)
SELECT * FROM main.silver.users VERSION AS OF 5;
SELECT * FROM main.silver.users TIMESTAMP AS OF '2026-01-10T12:00:00Z';

-- See table transaction history
DESCRIBE HISTORY main.silver.users;

-- Restore a table to a previous version
RESTORE TABLE main.silver.users TO VERSION AS OF 5;
````

---

### Page 2: ETL, ELT, and Data Ingestion

**Reading & Writing Data (PySpark)**
````python
# Read from various sources
df_json = spark.read.format("json").load("/mnt/raw/landing/users.json")
df_delta = spark.read.table("main.bronze.orders")

# Basic transformations
from pyspark.sql.functions import col, upper
transformed_df = df_json.filter(col("user_id").isNotNull()) \
                        .withColumn("name", upper(col("name")))

# Write to a Delta table
(transformed_df.write
  .format("delta")
  .mode("overwrite") # Common modes: append, overwrite, errorifexists
  .saveAsTable("main.silver.users"))
````

**Auto Loader (for Incremental Ingestion)**
The best way to incrementally and idempotently process files from cloud storage into a Delta table.
- **Key Features**: Schema inference and evolution, checkpointing to track ingested files.
- Use `cloudFiles` format.

````python
# Use Auto Loader in a streaming read
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", "/mnt/schemas/users") # For schema tracking
  .load("/mnt/raw/landing/users/"))

(df.writeStream
  .format("delta")
  .option("checkpointLocation", "/mnt/checkpoints/users") # For progress tracking
  .trigger(availableNow=True) # Process all available data then stop (batch-like)
  .toTable("main.bronze.users"))
````

**Delta Live Tables (DLT)**
A declarative framework for building reliable data pipelines.
- **Declarative**: Define the transformations; DLT manages the execution graph, infrastructure, and data quality.
- **Data Quality**: Use *expectations* to define and handle data quality rules.

````python
import dlt
from pyspark.sql.functions import *

@dlt.table(comment="Raw user data from cloud storage.")
def users_bronze():
  return (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .load("/mnt/raw/users"))

@dlt.table(comment="Cleaned, validated user data.")
@dlt.expect_or_drop("valid_user_id", "user_id IS NOT NULL")
def users_silver():
  return dlt.read_stream("users_bronze").select("user_id", "name", "email")
````

**Change Data Capture (CDC) with `MERGE`**
The `MERGE` statement is used to perform "upserts" (updates, inserts, and deletes) in a single atomic operation. Essential for applying changes to Silver tables.
````sql
MERGE INTO main.silver.customers target
USING main.bronze.customer_updates source
ON target.customer_id = source.customer_id
WHEN MATCHED AND source.is_deleted = true THEN
  DELETE
WHEN MATCHED THEN
  UPDATE SET target.address = source.address, target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (customer_id, address, created_at)
  VALUES (source.customer_id, source.address, current_timestamp());
````

---

### Page 3: Workflows, Governance & Operations

**Databricks Workflows (Jobs)**
The tool for orchestrating and scheduling production pipelines.
- **Tasks**: A job consists of one or more tasks (e.g., Notebook, Python Script, DLT Pipeline, SQL query).
- **Dependencies**: Define the execution order of tasks to create a DAG (Directed Acyclic Graph).
- **Parameters**: Pass parameters between tasks and to jobs at runtime.
- **Alerts & Retries**: Configure notifications for failures/success and set automatic retry policies.

**Performance Tuning**
- **`OPTIMIZE`**: Compacts small files into larger ones to improve read performance.
- **`ZORDER`**: A technique for co-locating related information in the same set of files. Use on columns frequently used in `WHERE` clauses.
- **`VACUUM`**: Removes files that are no longer referenced by a Delta table and are older than the retention threshold (default 7 days). **Warning**: `VACUUM` removes the ability to time travel past the retention period.
- **Partitioning**: Physically divide data based on column values (e.g., `date`). Speeds up queries that filter on the partition key. Best for low-cardinality columns.

````sql
-- Optimize and Z-Order a table
OPTIMIZE main.gold.sales_aggregates ZORDER BY (product_id);

-- Clean up old files
VACUUM main.gold.sales_aggregates RETAIN 168 HOURS; -- Retain 7 days
````

**Unity Catalog (UC) for Governance**
- **Permissions**: Use standard SQL `GRANT` and `REVOKE` to manage access to catalogs, schemas, tables, and views.
- **Ownership**: Every object has an owner.
- **External Locations**: A UC object that maps a storage path (e.g., `s3://my-bucket/data`) to a `STORAGE CREDENTIAL`, allowing secure access to cloud storage.

````sql
-- Grant usage on a schema and select on a table
GRANT USAGE ON SCHEMA main.silver TO `analysts`;
GRANT SELECT ON TABLE main.silver.customers TO `analysts`;
````

**Custom ETL Logging (Pattern from your SQL)**
The logging pattern in your `sp_LogDataFlow.sql` file is common. Here's how you might implement it in a Databricks notebook using a Delta table.

````python
def log_data_flow(status, data_flow_sk, load_execution_id, message=None, insert_row_count=None):
  from pyspark.sql.functions import lit, current_timestamp

  log_table = "main.monitoring.cdc_dataflowlog"
  now = current_timestamp()

  if status == 'Started':
    log_df = spark.createDataFrame([
        (data_flow_sk, load_execution_id, status)
    ], ["DataFlow_sk", "LoadExecutionId", "Status"])
    
    (log_df.withColumn("StartedAt", now)
           .write.format("delta").mode("append").saveAsTable(log_table))
  else:
    update_condition = f"LoadExecutionId = '{load_execution_id}' AND DataFlow_sk = '{data_flow_sk}'"
    update_values = {
      "FinishedAt": now,
      "Status": lit(status),
      "Message": lit(message),
      "InsertRowCount": lit(insert_row_count)
    }
    
    from delta.tables import DeltaTable
    delta_log_table = DeltaTable.forName(spark, log_table)
    delta_log_table.update(condition=update_condition, set=update_values)

# --- Usage in a data flow notebook ---
# log_data_flow('Started', 'dim_customer_load', dbutils.widgets.get('run_id'))
# ... ETL logic ...
# result_df.write.saveAsTable(...)
# log_data_flow('Finished', 'dim_customer_load', dbutils.widgets.get('run_id'), 'Success', result_df.count())
````